# Probabilidad, MLE y MAP: cómo un modelo aprende

**Ciencia de Datos, Sección A** · Sesión 5 · 4 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install numpy matplotlib`

## 1. Repaso express: Bayes con el filtro de spam

$$P(H \mid D) = \frac{P(D \mid H)\,P(H)}{P(D)}$$

Un correo trae la palabra *oferta*. ¿Es spam?

In [1]:
p_spam = 0.20                 # prior
p_oferta_dado_spam = 0.60
p_oferta_dado_no = 0.05

num = p_oferta_dado_spam * p_spam
den = num + p_oferta_dado_no * (1 - p_spam)
print(num / den)              # 0.75

0.75


## 2. Simular para entender

Cuando la fórmula no da intuición, simulen: es gratis y no se equivoca.

In [2]:
import numpy as np

rng = np.random.default_rng(42)

x = rng.binomial(1, 0.3, size=1000)   # moneda con p=0.3
print(x.mean())                       # ~0.30

alturas = rng.normal(170, 10, size=1000)
print(alturas.mean(), alturas.std())  # ~170, ~10

0.304
169.22117451458604 10.139672980433375


In [ ]:
import matplotlib.pyplot as plt

AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(alturas, bins=30, density=True, color=AZUL, alpha=0.35,
        edgecolor="white", label="1000 alturas simuladas")
xs = np.linspace(135, 205, 300)
pdf = np.exp(-(xs - 170)**2 / (2 * 10**2)) / (10 * np.sqrt(2 * np.pi))
ax.plot(xs, pdf, color=ROJO, lw=2, label="Normal(170, 10) verdadera")
ax.set_xlabel("altura (cm)")
ax.set_ylabel("densidad")
ax.set_title("La distribución que generó los datos")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

In [3]:
# Ley de los grandes números: el promedio converge a p
n = np.arange(1, 1001)
promedio = np.cumsum(x) / n

print(promedio[9], promedio[99], promedio[999])

0.5 0.27 0.304


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(n, promedio, color=AZUL, lw=1.8, label="promedio acumulado")
ax.axhline(0.3, color=GRIS, ls="--", lw=1.5, label="p verdadero = 0.3")
ax.set_xscale("log")
ax.set_xlabel("número de lanzamientos (escala log)")
ax.set_ylabel("proporción de caras")
ax.set_title("Ley de los grandes números: el promedio converge a p")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

## 3. La verosimilitud

Misma fórmula, leída con la variable contraria fija:

- **Probabilidad**: conozco $\theta$, pregunto por los datos
- **Verosimilitud**: tengo los datos, pregunto por $\theta$

$$L(\theta) = P(\text{datos} \mid \theta)$$

Lanzo 10 veces y salen 7 caras. ¿Qué `p` hace esos datos más plausibles?

In [4]:
k, n = 7, 10
p = np.linspace(0.01, 0.99, 99)
L = p**k * (1 - p)**(n - k)

print(p[np.argmax(L)])   # 0.70 = k/n

0.7000000000000001


In [ ]:
p_hat = p[np.argmax(L)]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(p, L, color=AZUL, lw=2)
ax.scatter(p[::6], L[::6], s=14, color=AZUL, zorder=3)
ax.axvline(p_hat, color=ROJO, ls="--", lw=1.5)
ax.annotate(f"máximo en p = {p_hat:.2f} = k/n",
            xy=(p_hat, L.max()), xytext=(0.12, L.max() * 0.85),
            color=ROJO,
            arrowprops=dict(arrowstyle="->", color=ROJO, lw=1.2))
ax.set_xlabel("p")
ax.set_ylabel("L(p)")
ax.set_title("Verosimilitud sobre una grilla de p (los puntos son la grilla)")
eje_limpio(ax)
plt.tight_layout()
plt.show()

## 4. MLE: máxima verosimilitud

Derivación para la moneda:

$$L(p) = p^k (1-p)^{n-k}$$
$$\ell(p) = k \log p + (n-k)\log(1-p)$$
$$\frac{d\ell}{dp} = \frac{k}{p} - \frac{n-k}{1-p} = 0 \;\Longrightarrow\; \hat{p} = \frac{k}{n}$$

Usamos el logaritmo porque convierte productos en sumas, evita el underflow numérico y no cambia el argmax.

In [5]:
k, n = 7, 10

def log_lik(p):
    return k*np.log(p) + (n-k)*np.log(1-p)

p = np.linspace(0.001, 0.999, 999)
ll = log_lik(p)
print(p[np.argmax(ll)], k / n)   # 0.700 en ambos

0.7000000000000001 0.7


In [ ]:
L = p**k * (1 - p)**(n - k)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True,
                         layout="constrained")
for ax, curva, nombre in [(axes[0], L, "L(p): producto"),
                          (axes[1], ll, "log L(p): suma")]:
    ax.plot(p, curva, color=AZUL, lw=2)
    ax.axvline(k / n, color=ROJO, ls="--", lw=1.5)
    ax.set_title(nombre)
    ax.set_xlabel("p")
    eje_limpio(ax)
axes[0].set_ylabel("L(p)")
axes[1].set_ylabel("log L(p)")
fig.suptitle("El log no cambia el argmax: ambas curvas pican en 0.70")
plt.show()

In [6]:
# Por qué log: multiplicar muchas probabilidades da 0 en float64
print(0.5 ** 2000)                    # 0.0 exacto
print(2000 * np.log(0.5))             # -1386.29, sin problema

0.0
-1386.2943611198905


### MLE para la Normal

$$\hat{\mu} = \frac{1}{n}\sum_i x_i, \qquad \hat{\sigma}^2 = \frac{1}{n}\sum_i (x_i - \hat{\mu})^2$$

Maximizar la verosimilitud de una Normal equivale a **minimizar la suma de errores al cuadrado**: ese es el puente con la regresión lineal del 18 de agosto.

In [7]:
d = rng.normal(170, 10, size=500)

mu_hat = d.mean()
sigma_hat = d.std()      # np.std divide entre n, como el MLE
print(mu_hat, sigma_hat)

169.92118300743516 10.123162414656978


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(d, bins=30, density=True, color=AZUL, alpha=0.35,
        edgecolor="white", label="500 datos")
xs = np.linspace(d.min() - 5, d.max() + 5, 300)
ajuste = (np.exp(-(xs - mu_hat)**2 / (2 * sigma_hat**2))
          / (sigma_hat * np.sqrt(2 * np.pi)))
ax.plot(xs, ajuste, color=ROJO, lw=2,
        label=f"MLE: Normal({mu_hat:.1f}, {sigma_hat:.1f})")
ax.axvline(mu_hat, color=ROJO, ls=":", lw=1.2)
ax.set_xlabel("altura (cm)")
ax.set_ylabel("densidad")
ax.set_title("La campana que el MLE ajusta sobre los datos")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

## 5. MAP: agregar un prior

$$P(\theta \mid D) \propto P(D \mid \theta) \times P(\theta)$$

Con un prior Beta($\alpha,\beta$) sobre `p`:

$$\hat{p}_{MAP} = \frac{k + \alpha - 1}{n + \alpha + \beta - 2}$$

Con Beta(2,2) queda $(k+1)/(n+2)$: como si hubiéramos visto una cara y un escudo imaginarios antes de empezar.

In [8]:
def mle(k, n):
    return k / n

def map_beta(k, n, a=2, b=2):
    return (k + a - 1) / (n + a + b - 2)

for k, n in [(2, 2), (20, 20), (700, 1000)]:
    print(n, round(mle(k, n), 3), round(map_beta(k, n), 3))

2 1.0 0.75
20 1.0 0.955
1000 0.7 0.7


In [ ]:
grilla = np.linspace(0.001, 0.999, 999)
a, b = 2, 2
prior = grilla**(a - 1) * (1 - grilla)**(b - 1)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharex=True,
                         layout="constrained")
for ax, (k, n) in zip(axes, [(7, 10), (700, 1000)]):
    log_lik = k * np.log(grilla) + (n - k) * np.log(1 - grilla)
    lik = np.exp(log_lik - log_lik.max())
    post = prior * lik
    dx = grilla[1] - grilla[0]
    for curva, color, ls, nombre in [
        (prior, GRIS, "--", f"prior Beta({a},{b})"),
        (lik, AZUL, "-", "verosimilitud"),
        (post, ROJO, "-", "posterior"),
    ]:
        curva = curva / (curva.sum() * dx)   # normalizada en la grilla
        ax.plot(grilla, curva, color=color, ls=ls, lw=2, label=nombre)
    ax.axvline(k / n, color=AZUL, ls=":", lw=1.2)
    ax.axvline((k + a - 1) / (n + a + b - 2), color=ROJO, ls=":", lw=1.2)
    ax.set_title(f"k = {k}, n = {n}")
    ax.set_xlabel("p")
    eje_limpio(ax)
axes[0].set_ylabel("densidad")
axes[0].legend(frameon=False)
fig.suptitle("El prior manda con pocos datos y se diluye con muchos")
plt.show()

### MAP es MLE regularizado

$$\log P(\theta \mid D) = \underbrace{\ell(\theta)}_{\text{ajusta datos}} + \underbrace{\log P(\theta)}_{\text{penaliza rarezas}} + c$$

| Prior sobre los coeficientes | Equivale a |
|---|---|
| Normal(0, $\tau^2$) | regularización **L2** (ridge) |
| Laplace(0, b) | regularización **L1** (lasso) |

Regularizar no es un truco de ingeniería: es poner un prior por escrito.

## 6. Ejercicios

Completen donde dice `# ¿Qué va aquí?`.

### Ejercicio 1: log-verosimilitud de Bernoulli

Encuentren el MLE recorriendo una grilla de valores de `p` con `np.argmax`.

In [ ]:
rng = np.random.default_rng(3)
datos = rng.binomial(1, 0.35, size=200)   # array de 0s y 1s

def log_lik_bernoulli(p, datos):
    # ¿Qué va aquí?
    # Pista: k = datos.sum(), n = len(datos)
    pass

grilla = np.linspace(0.001, 0.999, 999)

# Verificación (descomenten al terminar):
# valores = np.array([log_lik_bernoulli(p, datos) for p in grilla])
# print(grilla[np.argmax(valores)], datos.mean())  # deben coincidir
# Para ver la curva y su máximo:
# plt.plot(grilla, valores)
# plt.axvline(grilla[np.argmax(valores)], ls="--", color=ROJO)

### Ejercicio 2: MLE de la Normal

Calculen los estimadores a mano y compárenlos con NumPy.

In [ ]:
muestra = rng.normal(170, 10, size=800)

# a) mu_hat: el promedio, sin usar .mean()
mu_hat = ...

# b) sigma2_hat: promedio de (x - mu_hat)**2
sigma2_hat = ...

# Verificación (descomenten al terminar):
# print(mu_hat, muestra.mean())
# print(np.sqrt(sigma2_hat), muestra.std())

# Para pensar: muestra.std(ddof=1) divide entre n-1. ¿Cuál es el MLE?

### Ejercicio 3: el prior se diluye

Muestren que el MAP se acerca al MLE conforme crecen los datos.

In [ ]:
def map_beta_ej(k, n, a=2, b=2):
    # ¿Qué va aquí?  La fórmula está en la lámina del prior Beta
    pass

verdadero_p = 0.7
rng = np.random.default_rng(5)

for n in [5, 20, 100, 1000, 10000]:
    k = rng.binomial(n, verdadero_p)
    # print(n, k / n, map_beta_ej(k, n))
    pass

# ¿A partir de qué n deja de importar el prior?

## Lo esencial de hoy

- La probabilidad modela el **ruido**; Bayes actualiza creencias
- $L(\theta)$ es la misma fórmula leída como función de $\theta$
- **MLE**: elegir el $\theta$ que hace los datos más plausibles ($\hat{p}=k/n$, $\hat{\mu}=\bar{x}$)
- Usamos $\log$ por sumas, estabilidad numérica y mismo argmax
- **MAP** = MLE + prior = MLE **regularizado** (ridge, lasso)
- Casi todo el curso será: elegir un modelo y maximizar su verosimilitud

**Próxima clase (jueves 6): Inferencia Estadística y A/B Testing.** Se asigna la HDT 2 (entrega: martes 18 de agosto).